In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import glob
import os

from microhhpy.spatial import calc_vertical_grid_2nd
from microhhpy.thermo import calc_moist_basestate, read_moist_basestate
from microhhpy.chem import Emission_input

## Emission input

MicroHH supports two methods for specifying emissions: Gaussian blobs defined in the `.ini` file, or 3D emission fields provided as binary input files.

This notebook demonstrates creating 3D emission fields either manually or using the `Emission_input` class.

### Manually create 3D emissions

Binary input files are the most easily created by saving 3D Numpy arrays in binary format. Emission units must be `base unit scalar / s`, e.g. `kg_co2 / kg_air / s` or `mol_co2 / mol_air / s`.

In [ ]:
itot = 64
jtot = 64
ktot_emiss = 32

co2_e = np.zeros((ktot_emiss, jtot, itot), dtype=np.float64)

co2_e[10,10,10] = np.pi

co2_e.tofile('co2_emission.0000000')

Problems arise when using `swbasestate=anelastic`, as the base state density is needed for unit conversions (e.g., `kg_co2 s-1` to `kg_co2 kg_air-1 s-1`). Obtain it either from the `rhoref.0000000` file generated by the MicroHH `init` phase or by calculating it with `microhhpy`. See grid_and_basestate.ipynb for details.

### 3D emissions using `microhhpy`

Alternatively, the `Emission_input` class from `microhhpy` supports adding sources as Gaussian blobs or point sources (single grid point).

In [ ]:
xsize = 300
ysize = 300
zsize = 300

itot = 300
jtot = 300
ktot = 300

dx = xsize / itot
dy = ysize / jtot
dz = zsize / ktot

x = np.arange(dx/2, xsize, dx)
y = np.arange(dy/2, ysize, dy)
z = np.arange(dz/2, zsize, dz)

# Calculate exact definition vertical grid.
# Not really necessary for equidistant grid, more important for e.g. stretched grids.
gd = calc_vertical_grid_2nd(z, zsize)

# Initial fields.
thl = 290 + 0.006 * z
qt  = np.zeros(ktot)
ps = 1e5

# Calculate base state model.
bs = calc_moist_basestate(thl, qt, ps, z, zsize)

In [ ]:
"""
Create emission input.
"""
times = np.array([0, 3600])     # Or simply np.array([0]) for non time-dependent emissions.

fields = ['s1']
emiss = Emission_input(fields, times, x, y, z, gd['dz'], bs['rho'])

for time in times:
    emiss.add_gaussian(field='s1', strength=1, time=time, x0=150, y0=150, z0=100, sigma_x=25, sigma_y=25, sigma_z=25, sw_vmr=True)
    emiss.add_gaussian(field='s1', strength=1, time=time, x0=50, y0=50, z0=50, sigma_x=10, sigma_y=10, sigma_z=10, sw_vmr=False)
    emiss.add_point(field='s1',    strength=0.01, time=time, x0=250, y0=200, z0=100, sw_vmr=False)

# The underlying arrays are exposed such that emissions can still be added manually:
print('Shape emission field = ', emiss.data['s1'].shape)

# Clip vertical extent to save space on disk and memory in MicroHH.
emiss.clip()

# Save as binary input for MicroHH.
# Resulting files are saved as `{path}/{name}_emission.{time:07d}`.
emiss.to_binary(path='.')

# Vertical extent emissions has to be provided in .ini as `source->ktot`, and is available as `emiss.kmax`.

In [ ]:
"""
Plot.
"""
plt.figure()
plt.pcolormesh(x, z[:emiss.kmax], emiss.data['s1'][0, :, :, :].sum(axis=1))
plt.colorbar()

In [ ]:
files = glob.glob('*00*')
for f in files:
    os.remove(f)